# SGLang 0.5.20 throughput test on 4xL4 (offline)

Source: `kaggle/src/bench_sglang.py`.

In [ ]:
import os, time
os.environ['ARC_T0'] = str(time.time())
os.environ['ARC_MODE'] = 'bench'

In [ ]:
open('/kaggle/working/solver.py', 'w').write("\"\"\"ARC Prize 2026 (ARC-AGI-2) offline solver: Qwen3.8-27B-FP8 on vLLM (TP=4, 4xL4), no tools.\n\nOne self-contained file; kaggle/build_notebook.py embeds it in the Kaggle notebook. Modes (env ARC_MODE):\n\n- submit:   full run on the competition test file (used automatically on the competition rerun).\n- fallback: write a valid fallback submission.json and exit (used for the commit run before submitting).\n- bench:    test run: provenance hashes, server start, long-context decode benchmark, short end-to-end run on\n            evaluation tasks with a near deadline, scoring against the evaluation solutions.\n            ARC_HASH=0 / ARC_BENCH_DECODE=0 skip the first two (validation run: ARC_E2E_TASKS, ARC_E2E_MIN).\n\nRobustness rules: a complete, valid submission.json is written before anything else and rewritten atomically\nafter every finished attempt; every failure path keeps the last valid file.\n\nPrompt, parser and Qwen raw-template rendering follow evals/arc_harness/arcprize.py and evals/run_qwen_generate.py\n(ARC Prize official prompt, tier xhigh, temp 1.0, top_p 0.95, top_k 20, forced final answer at the budget).\n\"\"\"\nimport asyncio\nimport glob\nimport hashlib\nimport json\nimport os\nimport re\nimport subprocess\nimport sys\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\n\nT0 = float(os.environ.get(\"ARC_T0\", time.time()))  # notebook start (set by the first cell)\nMODE = os.environ.get(\"ARC_MODE\", \"submit\")\nWORK = \"/kaggle/working\"\nOUT_SUB = os.path.join(WORK, \"submission.json\")\nLOG_DIR = os.path.join(WORK, \"arc_logs\")\nPORT = 8000\nBASE = f\"http://127.0.0.1:{PORT}\"\nSERVED = \"qwen3.8-27b\"\n\n# ---------------------------------------------------------------- settings (env overridable)\n\ndef envf(name, default):\n    return type(default)(os.environ.get(name, default))\n\nFORCE_AT_S = envf(\"ARC_FORCE_AT_S\", 11.0 * 3600)    # force final answers for all in-flight attempts\nHARD_STOP_S = envf(\"ARC_HARD_STOP_S\", 11.6 * 3600)  # stop everything, final write\nB_MAX1 = envf(\"ARC_B_MAX1\", 96000)                  # attempt-1 budget cap (thinking + answer)\nB_MIN1 = envf(\"ARC_B_MIN1\", 16000)\nB_MAX2 = envf(\"ARC_B_MAX2\", 64000)\nB_MIN2 = envf(\"ARC_B_MIN2\", 32000)                  # attempt 2 is launched only if it can get this much\nFINAL_TOKENS = envf(\"ARC_FINAL_TOKENS\", 3000)\nRATE_PRIOR = envf(\"ARC_RATE_PRIOR\", 350.0)          # tok/s before the run measures its own rate\nMAX_MODEL_LEN = envf(\"ARC_MAX_MODEL_LEN\", 131072)\nMAX_SEQS = envf(\"ARC_MAX_SEQS\", 64)\nCONCURRENCY = envf(\"ARC_CONCURRENCY\", 0)            # 0 = from KV capacity\nTOKENS_PER_SEQ_KV = envf(\"ARC_KV_PER_SEQ\", 30000)   # KV tokens assumed per sequence when sizing concurrency\nMTP = envf(\"ARC_MTP\", 0)                            # speculative tokens (0 = off)\nTIER = os.environ.get(\"ARC_TIER\", \"xhigh\")\nTEMPERATURE = envf(\"ARC_TEMPERATURE\", 1.0)\nGPU_UTIL = envf(\"ARC_GPU_UTIL\", 0.95)\n\nPROMPT_TEMPLATE = \"You are participating in a puzzle solving competition. You are an expert at solving puzzles.\\n\\nBelow is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation in the training examples that maps the input to the output, then apply that pattern to the test input to give a final output.\\n\\nRespond in the format of the training output examples\\n\\n--Training Examples--\\n{training_examples}\\n--End of Training Examples--\\n\\n--Test Input--\\n{test_input}\\n--End of Test Input--\\n\\nYour response:\"  # noqa: F821  (ARC Prize official prompt, injected by build_notebook.py)\n\nCLOSE = \"</think>\"\nFORCE_TEXT = \"\\n\\nI have to give my final answer now.\\n</think>\\n\\n\"\nLOW_YIELD = 20\nTIER_SYSTEM = {\n    \"xhigh\": \"Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, \"\n             \"consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final \"\n             \"answer.\",\n    \"medium\": None,\n}\n\n\ndef log(*a):\n    el = time.time() - T0\n    print(f\"[{el / 3600:5.2f}h {time.strftime('%H:%M:%S')}]\", *a, flush=True)\n\n\n# ---------------------------------------------------------------- prompt / parse (verbatim from arc_harness)\n\ndef build_prompt(train, test_input):\n    ex = \"\"\n    for i, pair in enumerate(train):\n        ex += f\"--Example {i}-- \\n\\n INPUT: \\n\\n\" + json.dumps(pair[\"input\"]) + \"\\n\\n\"\n        ex += \"OUTPUT: \\n\\n\" + json.dumps(pair[\"output\"]) + \"\\n\\n\"\n    return PROMPT_TEMPLATE.format(training_examples=ex, test_input=json.dumps(test_input))\n\n\ndef render_prompt(user_content, tier=TIER):\n    s = \"\"\n    if TIER_SYSTEM.get(tier):\n        s += f\"<|im_start|>system\\n{TIER_SYSTEM[tier]}<|im_end|>\\n\"\n    return s + f\"<|im_start|>user\\n{user_content.strip()}<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\"\n\n\ndef answer_section(gen):\n    i = gen.rfind(CLOSE)\n    return None if i < 0 else gen[i + len(CLOSE):]\n\n\ndef is_grid(x):\n    return isinstance(x, list) and len(x) > 0 and all(isinstance(r, list) for r in x)\n\n\ndef extract_from_boxed(s):\n    m = re.search(r\"\\\\boxed\\{(.*?)\\}\", s, re.DOTALL)\n    if m:\n        try:\n            g = json.loads(m.group(1).strip())\n            return g if is_grid(g) else None\n        except json.JSONDecodeError:\n            return None\n    return None\n\n\ndef backscan_json_parser(s):\n    end = max(s.rfind(\"]\"), s.rfind(\"}\"))\n    if end < 0:\n        return None\n    close = s[end]\n    open_ = \"[\" if close == \"]\" else \"{\"\n    depth = 1\n    for i in range(end - 1, -1, -1):\n        depth += (s[i] == close) - (s[i] == open_)\n        if depth == 0:\n            try:\n                g = json.loads(s[i:end + 1])\n                return g if is_grid(g) else None\n            except json.JSONDecodeError:\n                return None\n    return None\n\n\ndef parse(s):\n    return extract_from_boxed(s or \"\") or backscan_json_parser(s or \"\")\n\n\ndef valid_grid(grid):\n    return (isinstance(grid, list) and 1 <= len(grid) <= 30\n            and all(isinstance(row, list) and 1 <= len(row) <= 30 for row in grid)\n            and len({len(row) for row in grid}) == 1\n            and all(type(v) is int and 0 <= v <= 9 for row in grid for v in row))\n\n\n# ---------------------------------------------------------------- paths\n\ndef find_one(pattern):\n    hits = sorted(glob.glob(pattern, recursive=True))\n    return hits[0] if hits else None\n\n\ndef comp_file(name):\n    return (find_one(f\"/kaggle/input/competitions/arc-prize-2026-arc-agi-2/{name}\")\n            or find_one(f\"/kaggle/input/arc-prize-2026-arc-agi-2/{name}\")\n            or find_one(f\"/kaggle/input/**/{name}\"))\n\n\ndef model_dir():\n    for pat in [\"/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/**/config.json\",\n                \"/kaggle/input/qwen3-8-27b-fp8-official/**/config.json\",\n                \"/kaggle/input/**/qwen3-8-27b-fp8*/**/config.json\"]:\n        p = find_one(pat)\n        if p:\n            return os.path.dirname(p)\n    raise FileNotFoundError(\"model dir\")\n\n\ndef wheel_dir():\n    p = (find_one(\"/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3/**/requirements.lock\")\n         or find_one(\"/kaggle/input/**/arc3-vllm-h100-wheelhouse-v3/**/requirements.lock\"))\n    if not p:\n        raise FileNotFoundError(\"wheelhouse\")\n    return os.path.dirname(p)\n\n\n# ---------------------------------------------------------------- submission\n\nclass Submission:\n    def __init__(self, tasks):\n        self.tasks = tasks\n        self.answers = {}  # (tid, pair) -> {attempt_index: grid}\n\n    def fallback(self, tid, p):\n        g = self.tasks[tid][\"test\"][p][\"input\"]\n        return g if valid_grid(g) else [[0]]\n\n    def build(self):\n        sub = {}\n        for tid, task in self.tasks.items():\n            lst = []\n            for p in range(len(task[\"test\"])):\n                got = [g for _, g in sorted(self.answers.get((tid, p), {}).items()) if valid_grid(g)]\n                a1 = got[0] if got else self.fallback(tid, p)\n                a2 = next((g for g in got[1:] if g != a1), None)\n                if a2 is None:\n                    a2 = self.fallback(tid, p) if a1 != self.fallback(tid, p) else a1\n                lst.append({\"attempt_1\": a1, \"attempt_2\": a2})\n            sub[tid] = lst\n        return sub\n\n    def write(self):\n        sub = self.build()\n        tmp = OUT_SUB + \".tmp\"\n        with open(tmp, \"w\") as f:\n            json.dump(sub, f)\n        os.replace(tmp, OUT_SUB)\n\n\ndef check_submission(path, tasks):\n    sub = json.load(open(path))\n    assert set(sub) == set(tasks), \"task ids differ\"\n    for tid, task in tasks.items():\n        assert len(sub[tid]) == len(task[\"test\"]), tid\n        for slot in sub[tid]:\n            assert valid_grid(slot[\"attempt_1\"]) and valid_grid(slot[\"attempt_2\"]), tid\n    return True\n\n\n# ---------------------------------------------------------------- environment\n\ndef sh(cmd, timeout=None):\n    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)\n    return r.returncode, (r.stdout or \"\") + (r.stderr or \"\")\n\n\ndef install_vllm():\n    try:\n        import vllm  # noqa: F401\n        log(\"vllm already installed\", vllm.__version__)\n        return\n    except Exception:\n        pass\n    wd = wheel_dir()\n    t = time.time()\n    code, out = sh(f\"{sys.executable} -m pip install --no-index --find-links {wd} -r {wd}/requirements.lock\",\n                   timeout=1800)\n    log(f\"pip install rc={code} in {time.time() - t:.0f}s\")\n    if code != 0:\n        print(out[-3000:])\n        raise RuntimeError(\"pip install failed\")\n    # The pinned py3.12 image ships a CPU torch 2.10.0, which satisfies the lock's torch==2.10.0, so pip keeps it.\n    # Force the wheelhouse's CUDA build (its nvidia-* deps were installed by the lock above).\n    code, out = sh(f\"{sys.executable} -m pip install --no-index --find-links {wd} --force-reinstall --no-deps \"\n                   f\"torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0\", timeout=1800)\n    log(f\"torch CUDA reinstall rc={code}\")\n    if code != 0:\n        print(out[-3000:])\n        raise RuntimeError(\"torch reinstall failed\")\n    code, out = sh(f\"{sys.executable} -c 'import torch, vllm; print(torch.__version__, torch.version.cuda, \"\n                   f\"torch.cuda.is_available(), torch.cuda.device_count(), vllm.__version__)'\", timeout=600)\n    log(\"torch/vllm:\", out.strip().splitlines()[-1] if out.strip() else out)\n\n\ndef file_hashes(paths, workers=16):\n    def h(p):\n        s256, s1 = hashlib.sha256(), hashlib.sha1()\n        size = os.path.getsize(p)\n        s1.update(f\"blob {size}\\0\".encode())\n        with open(p, \"rb\") as f:\n            while True:\n                b = f.read(1 << 24)\n                if not b:\n                    break\n                s256.update(b)\n                s1.update(b)\n        return p, {\"size\": size, \"sha256\": s256.hexdigest(), \"git_sha1\": s1.hexdigest()}\n    with ThreadPoolExecutor(workers) as ex:\n        return dict(ex.map(h, paths))\n\n\ndef preload(mdir, workers=16):\n    files = [os.path.join(r, f) for r, _, fs in os.walk(mdir) for f in fs]\n\n    def rd(p):\n        with open(p, \"rb\") as f:\n            while f.read(1 << 26):\n                pass\n    t = time.time()\n    with ThreadPoolExecutor(workers) as ex:\n        list(ex.map(rd, files))\n    log(f\"preloaded {len(files)} model files in {time.time() - t:.0f}s\")\n\n\ndef launch_server(mdir):\n    attn = \"TRITON_ATTN\"  # FlashInfer fp8-KV kernels need SM90+; L4 is SM89\n    cmd = [sys.executable, \"-m\", \"vllm.entrypoints.openai.api_server\", \"--model\", mdir,\n           \"--served-model-name\", SERVED, \"--host\", \"127.0.0.1\", \"--port\", str(PORT),\n           \"--tensor-parallel-size\", \"4\", \"--gpu-memory-utilization\", str(GPU_UTIL),\n           \"--max-model-len\", str(MAX_MODEL_LEN), \"--max-num-seqs\", str(MAX_SEQS),\n           \"--kv-cache-dtype\", \"fp8\", \"--attention-backend\", attn, \"--enable-prefix-caching\",\n           \"--trust-remote-code\", \"--no-async-scheduling\"]\n    if MTP > 0:\n        cmd += [\"--speculative-config\", json.dumps({\"method\": \"mtp\", \"num_speculative_tokens\": MTP})]\n    env = os.environ.copy()\n    env.update(PYTORCH_ALLOC_CONF=\"expandable_segments:True\", VLLM_FORCE_ATTN_BACKEND=attn,\n               VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=\"1\", TOKENIZERS_PARALLELISM=\"false\",\n               VLLM_LOGGING_LEVEL=\"INFO\")\n    real = \"/usr/local/nvidia/lib64\"\n    if os.path.exists(real + \"/libcuda.so\"):\n        env[\"LD_LIBRARY_PATH\"] = real + \":\" + env.get(\"LD_LIBRARY_PATH\", \"\")\n        env[\"LD_PRELOAD\"] = real + \"/libcuda.so\"\n    log(\"server cmd:\", \" \".join(cmd))\n    path = os.path.join(LOG_DIR, \"vllm_server.log\")\n    fh = open(path, \"w\")\n    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,\n                            start_new_session=True, env=env)\n    return proc, path\n\n\ndef server_log_tail(path, n=60):\n    try:\n        return \"\".join(open(path, errors=\"ignore\").readlines()[-n:])\n    except Exception:\n        return \"\"\n\n\ndef wait_ready(proc, path, timeout):\n    import urllib.request\n    t = time.time()\n    while time.time() - t < timeout:\n        if proc.poll() is not None:\n            log(f\"server exited rc={proc.returncode}\\n\" + server_log_tail(path))\n            return False\n        try:\n            with urllib.request.urlopen(BASE + \"/v1/models\", timeout=5) as r:\n                if r.status == 200:\n                    log(f\"server ready in {time.time() - t:.0f}s\")\n                    return True\n        except Exception:\n            pass\n        time.sleep(5)\n    log(\"server not ready in time\\n\" + server_log_tail(path))\n    return False\n\n\ndef kv_capacity(path):\n    txt = open(path, errors=\"ignore\").read()\n    m = re.findall(r\"GPU KV cache size: ([\\d,]+) tokens\", txt)\n    c = re.findall(r\"Maximum concurrency for ([\\d,]+) tokens per request: ([\\d.]+)x\", txt)\n    kv = int(m[-1].replace(\",\", \"\")) if m else None\n    log(\"KV cache tokens:\", kv, \"| max concurrency lines:\", c[-1:] if c else None)\n    return kv\n\n\n# ---------------------------------------------------------------- client\n\nclass Client:\n    def __init__(self):\n        import httpx\n        self.last_prompt_tokens = None\n        self.http = httpx.AsyncClient(base_url=BASE, timeout=httpx.Timeout(None, connect=30.0),\n                                      limits=httpx.Limits(max_connections=512, max_keepalive_connections=512))\n\n    async def count(self, prompt):\n        try:\n            r = await self.http.post(\"/tokenize\", json={\"model\": SERVED, \"prompt\": prompt})\n            return int(r.json()[\"count\"])\n        except Exception:\n            return int(len(prompt) / 2.5)\n\n    async def stream(self, prompt, max_tokens, deadline, on_tokens):\n        \"\"\"Streamed completion. Returns (text, completion_tokens, finish_reason); finish 'deadline' if cut off.\n\n        Breaking out of the stream closes the connection, which makes vLLM abort the request.\n        \"\"\"\n        body = {\"model\": SERVED, \"prompt\": prompt, \"max_tokens\": max_tokens, \"temperature\": TEMPERATURE,\n                \"top_p\": 0.95, \"top_k\": 20, \"stream\": True, \"skip_special_tokens\": False,\n                \"stream_options\": {\"include_usage\": True, \"continuous_usage_stats\": True}}\n        text, ntok, finish = [], 0, None\n        async with self.http.stream(\"POST\", \"/v1/completions\", json=body) as r:\n            if r.status_code != 200:\n                raise RuntimeError(f\"HTTP {r.status_code}: {(await r.aread())[:300]!r}\")\n            async for line in r.aiter_lines():\n                if not line.startswith(\"data: \"):\n                    continue\n                data = line[6:]\n                if data.strip() == \"[DONE]\":\n                    break\n                d = json.loads(data)\n                u = d.get(\"usage\")\n                got_text = False\n                for c in d.get(\"choices\") or []:\n                    text.append(c.get(\"text\") or \"\")\n                    got_text = got_text or bool(c.get(\"text\"))\n                    if c.get(\"finish_reason\"):\n                        finish = c[\"finish_reason\"]\n                if u and u.get(\"completion_tokens\") is not None:\n                    self.last_prompt_tokens = u.get(\"prompt_tokens\") or self.last_prompt_tokens\n                    on_tokens(u[\"completion_tokens\"] - ntok)\n                    ntok = u[\"completion_tokens\"]\n                elif got_text:  # server without continuous usage stats: ~1 token per chunk until usage arrives\n                    on_tokens(1)\n                    ntok += 1\n                if time.time() > deadline:\n                    finish = \"deadline\"\n                    break\n        return \"\".join(text), ntok, finish or \"stop\"\n\n\n# ---------------------------------------------------------------- scheduler\n\nclass Scheduler:\n    def __init__(self, tasks, sub, client, force_at, hard_stop, concurrency, attempts=2, tag=\"run\"):\n        self.tasks, self.sub, self.client = tasks, sub, client\n        self.force_at, self.hard_stop = force_at, hard_stop\n        self.concurrency, self.attempts = concurrency, attempts\n        self.pending1 = []\n        self.pending2 = []\n        self.inflight = {}  # job -> [budget, used]\n        self.tokens = 0\n        self.rate_samples = []  # (time, total tokens)\n        self.gen_start = None\n        self.records = []\n        self.rec_path = os.path.join(LOG_DIR, f\"attempts_{tag}.jsonl\")\n        self.launch_margin = 120  # no new attempts in the last 2 min before forcing\n        self.retries = {}\n        self.server_down = lambda: False  # set by main(): True while the vLLM server is dead or restarting\n\n    def on_tokens(self, n):\n        self.tokens += n\n\n    def rate(self):\n        now = time.time()\n        self.rate_samples.append((now, self.tokens))\n        self.rate_samples = [s for s in self.rate_samples if now - s[0] < 1800] or [(now, self.tokens)]\n        t_old, k_old = self.rate_samples[0]\n        if self.gen_start is None or now - self.gen_start < 900 or now - t_old < 300:\n            return RATE_PRIOR\n        return max(50.0, (self.tokens - k_old) / (now - t_old))\n\n    def capacity_left(self):\n        cap = self.rate() * max(0.0, self.force_at - time.time())\n        return cap - sum(max(0, b - u) for b, u in self.inflight.values())\n\n    def next_job(self):\n        \"\"\"(job, budget) or None. Attempt-1 jobs first; attempt 2 only with spare capacity.\"\"\"\n        if self.pending1:\n            job = self.pending1.pop(0)\n            share = self.capacity_left() / (len(self.pending1) + 1)\n            return job, int(min(B_MAX1, max(B_MIN1, share)))\n        while self.pending2:\n            job = self.pending2.pop(0)\n            share = self.capacity_left() / (len(self.pending2) + 1)\n            if share < B_MIN2:\n                continue\n            return job, int(min(B_MAX2, share))\n        return None\n\n    async def attempt(self, job, budget):\n        tid, p, a = job\n        task = self.tasks[tid]\n        raw = render_prompt(build_prompt(task[\"train\"], task[\"test\"][p][\"input\"]))\n        ptok = await self.client.count(raw)\n        rec = {\"task_id\": tid, \"pair\": p, \"attempt\": a, \"budget\": budget, \"prompt_tokens\": ptok}\n        self.inflight[job] = [budget, 0]\n        gen, used, conts, streak, forced, force_reason, after_eos, chunks = \"\", 0, 0, 0, False, None, False, []\n        t0 = time.time()\n        try:\n            while True:\n                room = MAX_MODEL_LEN - ptok - used - 64\n                if forced:\n                    mt, dl = min(FINAL_TOKENS, room), self.hard_stop\n                else:\n                    mt, dl = min(budget - used, room - FINAL_TOKENS), self.force_at\n                if mt <= 0 and not forced:\n                    forced, force_reason = True, \"budget\"\n                    gen += FORCE_TEXT\n                    continue\n                if mt <= 0:\n                    break\n                text, ntok, finish = await self.client.stream(raw + gen, mt, dl, self.on_tokens)\n                gen += text\n                used += ntok\n                self.inflight[job][1] = used\n                chunks.append([ntok, finish])\n                if after_eos:\n                    streak = streak + 1 if ntok < LOW_YIELD else 0\n                if forced:\n                    break\n                if answer_section(gen) is not None:\n                    if finish == \"length\" and used < budget and room - ntok > 64:\n                        continue\n                    break\n                if finish == \"deadline\" or used >= budget or finish == \"length\":\n                    forced, force_reason = True, (\"deadline\" if finish == \"deadline\" else \"budget\")\n                    gen += FORCE_TEXT\n                    continue\n                # EOS mid-thought: continue, force after two low-yield continuations in a row\n                if streak >= 2:\n                    forced, force_reason = True, \"low_yield_eos\"\n                    gen += FORCE_TEXT\n                    continue\n                conts += 1\n                after_eos = True\n        except Exception as exc:\n            rec[\"error\"] = f\"{type(exc).__name__}: {exc}\"[:400]\n        finally:\n            self.inflight.pop(job, None)\n        content = answer_section(gen)\n        pred = parse(content) if content is not None else None\n        rec.update(tokens=used, continuations=conts, forced=forced, force_reason=force_reason, chunks=chunks,\n                   elapsed_s=round(time.time() - t0), answered=pred is not None and valid_grid(pred), pred=pred,\n                   tail=(content or gen[-300:])[-300:])\n        with open(os.path.join(LOG_DIR, f\"trace_{tid}_{p}_{a}.txt\"), \"w\") as f:\n            f.write(gen)\n        return rec\n\n    async def worker(self, wid):\n        while time.time() < self.force_at - self.launch_margin:\n            nj = self.next_job()\n            if nj is None:\n                return\n            job, budget = nj\n            rec = await self.attempt(job, budget)\n            if \"error\" in rec and not rec[\"answered\"] and self.server_down():\n                # server crashed: requeue the job (at most twice) and wait for the watchdog to restart it\n                self.retries[job] = self.retries.get(job, 0) + 1\n                if self.retries[job] <= 2:\n                    (self.pending1 if job[2] == 0 else self.pending2).insert(0, job)\n                    log(f\"server down; requeued {job}, waiting\")\n                    while self.server_down() and time.time() < self.force_at - self.launch_margin:\n                        await asyncio.sleep(15)\n                    continue\n            self.records.append(rec)\n            with open(self.rec_path, \"a\") as f:\n                f.write(json.dumps(rec) + \"\\n\")\n            tid, p, a = job\n            if rec[\"answered\"]:\n                self.sub.answers.setdefault((tid, p), {})[a] = rec[\"pred\"]\n                self.sub.write()\n            if a == 0 and self.attempts > 1:\n                self.pending2.append((tid, p, 1))\n            log(f\"done {tid}.{p}#{a} answered={rec['answered']} tok={rec['tokens']} budget={budget} \"\n                f\"forced={rec['forced']}/{rec['force_reason']} \"\n                f\"{rec['elapsed_s']}s {rec.get('error', '')}\")\n\n    async def monitor(self):\n        last_t, last_k = time.time(), 0\n        while True:\n            await asyncio.sleep(120)\n            now = time.time()\n            r = (self.tokens - last_k) / (now - last_t)\n            last_t, last_k = now, self.tokens\n            log(f\"progress: {len(self.records)} attempts done, in flight {len(self.inflight)}, pending1 \"\n                f\"{len(self.pending1)}, pending2 {len(self.pending2)}, tokens {self.tokens}, window {r:.0f} tok/s, \"\n                f\"rate est {self.rate():.0f} tok/s, avg {self.tokens / max(1, now - self.gen_start):.0f} tok/s\")\n\n    async def run(self, jobs1):\n        self.pending1 = list(jobs1)\n        self.gen_start = time.time()\n        mon = asyncio.create_task(self.monitor())\n        try:\n            await asyncio.wait_for(asyncio.gather(*(self.worker(i) for i in range(self.concurrency))),\n                                   timeout=max(60, self.hard_stop - time.time()))\n        except asyncio.TimeoutError:\n            log(\"hard stop reached; cancelling in-flight attempts\")\n        mon.cancel()\n        el = time.time() - self.gen_start\n        log(f\"scheduler finished: {len(self.records)} attempts, {self.tokens} tokens in {el / 60:.1f} min \"\n            f\"({self.tokens / max(1, el):.0f} tok/s)\")\n\n\ndef order_jobs(tasks):\n    \"\"\"Attempt-1 jobs, shortest prompts first (approximated by character length).\"\"\"\n    jobs = [(tid, p, 0) for tid, t in tasks.items() for p in range(len(t[\"test\"]))]\n    size = {tid: len(json.dumps(t[\"train\"])) + len(json.dumps(t[\"test\"])) for tid, t in tasks.items()}\n    return sorted(jobs, key=lambda j: (size[j[0]], j[0], j[1]))\n\n\n# ---------------------------------------------------------------- benchmark (test runs only)\n\nasync def bench_decode(client, prefixes, n_seqs, gen_tokens, tasks, max_chars=None, n_unique=None):\n    \"\"\"Decode throughput at long context: real ARC prompts + real Qwen trace prefixes, n_seqs in parallel.\n\n    n_unique: number of distinct prompts (sequence i uses prompt i % n_unique), so repeats can hit a prefix cache.\n    \"\"\"\n    total = {\"tok\": 0}\n    prompts = []\n    for i in range(n_seqs):\n        pr = prefixes[i % (n_unique or len(prefixes))]\n        t = tasks[pr[\"task_id\"]]\n        prompts.append(render_prompt(build_prompt(t[\"train\"], t[\"test\"][pr[\"pair\"]][\"input\"]))\n                       + pr[\"text\"][:max_chars])\n    ctx = [await client.count(p) for p in prompts]\n    log(f\"bench: {n_seqs} seqs, context tokens mean {sum(ctx) / len(ctx):.0f} min {min(ctx)} max {max(ctx)}\")\n    first = {}\n    t_start = time.time()\n\n    def cb(n):\n        total[\"tok\"] += n\n\n    async def one(i):\n        def cbi(n):\n            if i not in first:\n                first[i] = time.time()\n            cb(n)\n        return await client.stream(prompts[i], gen_tokens, time.time() + 3600, cbi)\n\n    samples = []\n\n    async def sampler():\n        while True:\n            await asyncio.sleep(10)\n            samples.append((time.time(), total[\"tok\"]))\n    s = asyncio.create_task(sampler())\n    res = await asyncio.gather(*(one(i) for i in range(n_seqs)), return_exceptions=True)\n    s.cancel()\n    t_end = time.time()\n    t_all_first = max(first.values()) if first else t_end\n    dec = [x for x in samples if x[0] >= t_all_first]\n    steady = (dec[-1][1] - dec[0][1]) / (dec[-1][0] - dec[0][0]) if len(dec) >= 2 else None\n    errs = [repr(r)[:200] for r in res if isinstance(r, Exception)]\n    out = {\"n_seqs\": n_seqs, \"ctx_mean\": sum(ctx) / len(ctx), \"gen_tokens\": gen_tokens, \"total_tokens\": total[\"tok\"],\n           \"wall_s\": t_end - t_start, \"prefill_phase_s\": t_all_first - t_start, \"steady_decode_tok_s\": steady,\n           \"overall_tok_s\": total[\"tok\"] / (t_end - t_start), \"errors\": errs}\n    log(\"bench result:\", json.dumps(out))\n    return out\n\n\ndef score(sub, solutions, tasks):\n    tot, pairs = 0.0, 0\n    for tid, t in tasks.items():\n        n = len(t[\"test\"])\n        ok = 0\n        for p in range(n):\n            s = sub[tid][p]\n            ok += solutions[tid][p] in (s[\"attempt_1\"], s[\"attempt_2\"])\n        tot += ok / n\n        pairs += n\n    return tot, len(tasks)\n\n\n# ---------------------------------------------------------------- main\n\ndef main():\n    os.makedirs(LOG_DIR, exist_ok=True)\n    log(f\"mode={MODE} T0={time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(T0))} UTC\")\n    rerun = bool(os.environ.get(\"KAGGLE_IS_COMPETITION_RERUN\"))\n    if MODE == \"bench\":\n        ch = comp_file(\"arc-agi_evaluation_challenges.json\")\n    else:\n        ch = comp_file(\"arc-agi_test_challenges.json\")\n    tasks = json.load(open(ch))\n    log(f\"challenges {ch}: {len(tasks)} tasks, {sum(len(t['test']) for t in tasks.values())} test inputs; \"\n        f\"rerun={rerun}\")\n    sub = Submission(tasks if MODE != \"bench\" else json.load(open(comp_file(\"arc-agi_test_challenges.json\"))))\n    sub.write()\n    log(\"fallback submission written\")\n    if MODE == \"fallback\":\n        return\n\n    mdir = model_dir()\n    log(\"model dir:\", mdir)\n    install_vllm()\n    if MODE == \"bench\" and os.environ.get(\"ARC_HASH\", \"1\") == \"1\":\n        files = sorted(os.path.join(r, f) for r, _, fs in os.walk(mdir) for f in fs)\n        wd = wheel_dir()\n        wheels = sorted(glob.glob(wd + \"/**/*\", recursive=True))\n        t = time.time()\n        hashes = {\"model\": {os.path.relpath(p, mdir): v for p, v in file_hashes(files).items()},\n                  \"wheels\": {os.path.relpath(p, wd): v for p, v in file_hashes([w for w in wheels if os.path.isfile(w)]).items()}}\n        json.dump(hashes, open(os.path.join(WORK, \"provenance_hashes.json\"), \"w\"), indent=1)\n        log(f\"hashed {len(hashes['model'])} model files + {len(hashes['wheels'])} wheel files in {time.time() - t:.0f}s\")\n    else:\n        preload(mdir)\n    proc, slog = launch_server(mdir)\n    if not wait_ready(proc, slog, timeout=1800):\n        raise RuntimeError(\"server failed to start\")\n    kv = kv_capacity(slog)\n    conc = CONCURRENCY or (min(MAX_SEQS, max(16, int((kv or 0) / TOKENS_PER_SEQ_KV))) if kv else 32)\n    log(f\"concurrency {conc} (max_num_seqs {MAX_SEQS}, MTP {MTP})\")\n\n    server = {\"proc\": proc, \"down\": False, \"restarts\": 0}\n\n    async def watchdog(force_at):\n        \"\"\"Restart the vLLM server once or twice if it dies, while there is time left to use it.\"\"\"\n        while True:\n            await asyncio.sleep(15)\n            if server[\"proc\"].poll() is None:\n                continue\n            server[\"down\"] = True\n            log(f\"vLLM server died (rc={server['proc'].returncode}):\\n\" + server_log_tail(slog, 40))\n            if server[\"restarts\"] >= 2 or force_at - time.time() < 45 * 60:\n                log(\"not restarting the server\")\n                return\n            server[\"restarts\"] += 1\n            os.rename(slog, slog + f\".crash{server['restarts']}\")\n            sh(\"pkill -9 -f 'vllm.entrypoints' ; sleep 5\", timeout=60)\n            server[\"proc\"], _ = launch_server(mdir)\n            ok = await asyncio.to_thread(wait_ready, server[\"proc\"], slog, 1800)\n            log(f\"server restart {'ok' if ok else 'FAILED'}\")\n            if not ok:\n                return\n            server[\"down\"] = False\n\n    async def go():\n        client = Client()\n        if MODE == \"bench\":\n            sol = json.load(open(comp_file(\"arc-agi_evaluation_solutions.json\")))\n            pf = find_one(\"/kaggle/input/**/bench_prefixes.json\")\n            prefixes = json.load(open(pf)) if pf else []\n            results = {\"kv_tokens\": kv, \"concurrency\": conc, \"mtp\": MTP}\n            if os.environ.get(\"ARC_BENCH_DECODE\", \"1\") == \"1\":\n                results[\"decode_long\"] = await bench_decode(client, prefixes, int(os.environ.get(\"ARC_BENCH_SEQS\", conc)),\n                                                            int(os.environ.get(\"ARC_BENCH_GEN\", 1500)), tasks)\n            json.dump(results, open(os.path.join(WORK, \"bench_results.json\"), \"w\"), indent=1)\n            # end-to-end: real scheduler on eval tasks with a deadline a few minutes away\n            e2e_min = float(os.environ.get(\"ARC_E2E_MIN\", 15))\n            ids = sorted(tasks)[:int(os.environ.get(\"ARC_E2E_TASKS\", 48))]\n            etasks = {t: tasks[t] for t in ids}\n            esub = Submission(etasks)\n            global OUT_SUB\n            keep = OUT_SUB\n            OUT_SUB = os.path.join(WORK, \"e2e_submission.json\")\n            sch = Scheduler(etasks, esub, client, time.time() + e2e_min * 60, time.time() + (e2e_min + 8) * 60,\n                            conc, attempts=1, tag=\"e2e\")\n            await sch.run(order_jobs(etasks))\n            esub.write()\n            check_submission(OUT_SUB, etasks)\n            sc, n = score(json.load(open(OUT_SUB)), sol, etasks)\n            OUT_SUB = keep\n            results[\"e2e\"] = {\"tasks\": n, \"score\": sc, \"attempts\": len(sch.records), \"tokens\": sch.tokens,\n                              \"answered\": sum(r[\"answered\"] for r in sch.records),\n                              \"forced\": sum(bool(r[\"forced\"]) for r in sch.records),\n                              \"errors\": sum(\"error\" in r for r in sch.records)}\n            log(\"e2e:\", json.dumps(results[\"e2e\"]))\n            json.dump(results, open(os.path.join(WORK, \"bench_results.json\"), \"w\"), indent=1)\n        else:\n            sch = Scheduler(tasks, sub, client, T0 + FORCE_AT_S, T0 + HARD_STOP_S, conc, attempts=2, tag=\"submit\")\n            sch.server_down = lambda: server[\"down\"] or server[\"proc\"].poll() is not None\n            wd = asyncio.create_task(watchdog(sch.force_at))\n            await sch.run(order_jobs(tasks))\n            wd.cancel()\n            sub.write()\n    asyncio.run(go())\n    log(\"server log tail:\\n\" + server_log_tail(slog, 15))\n\n\nif __name__ == \"__main__\":\n    try:\n        main()\n    except BaseException as exc:  # keep whatever submission.json exists\n        import traceback\n        traceback.print_exc()\n        log(f\"FATAL {type(exc).__name__}: {exc}\")\n    finally:\n        try:\n            tasks_path = comp_file(\"arc-agi_test_challenges.json\")\n            check_submission(OUT_SUB, json.load(open(tasks_path)))\n            log(\"submission.json valid\")\n        except Exception as exc:\n            log(f\"submission check failed ({exc}); rewriting fallback\")\n            Submission(json.load(open(comp_file(\"arc-agi_test_challenges.json\")))).write()\n        os._exit(0)\n")
open('/kaggle/working/bench_sglang.py', 'w').write("\"\"\"Offline SGLang throughput test on Kaggle 4xL4 (E013). Runs next to solver.py (imports its helpers).\n\nSteps, time-boxed by ARC_BENCH_STOP_MIN (default 47 min from notebook start; the push also sets a session timeout):\n1. Install SGLang 0.5.20 from the attached wheel source (chankhavu/sglang-0-5-20-py312-cu130) with --no-index,\n   pinned to its requirements.lock. Print torch/CUDA/driver info.\n2. Config A, radix cache ON: start server (TP=4, FP8 weights, fp8_e4m3 KV), record startup time and the memory pool\n   sizes from the log; decode benchmark with 16 sequences at ~40k context (same prompts as the vLLM bench), then 32\n   sequences reusing those 16 prompts (radix hits, so little extra prefill).\n3. Config B, radix cache OFF: restart, record pool sizes; short decode benchmark (16 sequences, ~10k context).\nResults: /kaggle/working/sglang_bench.json, server logs in /kaggle/working/arc_logs/.\n\"\"\"\nimport asyncio\nimport glob\nimport json\nimport os\nimport re\nimport subprocess\nimport sys\nimport time\n\nsys.path.insert(0, \"/kaggle/working\")\nimport solver as S  # noqa: E402\n\nSTOP = S.T0 + float(os.environ.get(\"ARC_BENCH_STOP_MIN\", 47)) * 60\nOUT = os.path.join(S.WORK, \"sglang_bench.json\")\nRESULTS = {\"configs\": {}}\n\n\ndef save():\n    json.dump(RESULTS, open(OUT, \"w\"), indent=1)\n\n\ndef left():\n    return STOP - time.time()\n\n\ndef wheel_dir_sglang():\n    p = S.find_one(\"/kaggle/input/**/sglang-0.5.20-*.whl\")\n    if not p:\n        raise FileNotFoundError(\"sglang wheel source not attached\")\n    return os.path.dirname(p)\n\n\ndef install():\n    wd = wheel_dir_sglang()\n    t = time.time()\n    code, out = S.sh(f\"{sys.executable} -m pip install --no-index --find-links {wd} -r {wd}/requirements.lock\",\n                     timeout=1800)\n    RESULTS[\"install_s\"] = round(time.time() - t)\n    S.log(f\"pip install (offline, lock) rc={code} in {RESULTS['install_s']}s\")\n    if code != 0:\n        print(out[-4000:])\n        raise RuntimeError(\"pip install failed\")\n    code, out = S.sh(f\"{sys.executable} -c 'import torch, sglang, flashinfer; print(torch.__version__, torch.version.cuda,\"\n                     f\" torch.cuda.is_available(), torch.cuda.device_count(), sglang.__version__, flashinfer.__version__)'\",\n                     timeout=600)\n    RESULTS[\"versions\"] = out.strip().splitlines()[-1] if out.strip() else out\n    _, smi = S.sh(\"nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader\", timeout=60)\n    RESULTS[\"gpus\"] = smi.strip()\n    S.log(\"versions:\", RESULTS[\"versions\"], \"| gpus:\", smi.strip().replace(\"\\n\", \"; \"))\n    save()\n\n\ndef launch(mdir, name, radix):\n    cmd = [sys.executable, \"-m\", \"sglang.launch_server\", \"--model-path\", mdir, \"--served-model-name\", S.SERVED,\n           \"--host\", \"127.0.0.1\", \"--port\", str(S.PORT), \"--tp-size\", \"4\", \"--kv-cache-dtype\", \"fp8_e4m3\",\n           \"--context-length\", \"131072\", \"--mem-fraction-static\", os.environ.get(\"ARC_SGL_MEM\", \"0.85\"),\n           \"--max-running-requests\", \"64\", \"--trust-remote-code\",\n           \"--attention-backend\", os.environ.get(\"ARC_SGL_ATTN\", \"triton\"), \"--log-level\", \"info\"]\n    if not radix:\n        cmd.append(\"--disable-radix-cache\")\n    env = os.environ.copy()\n    env.update(HF_HUB_OFFLINE=\"1\", TRANSFORMERS_OFFLINE=\"1\", TOKENIZERS_PARALLELISM=\"false\",\n               PYTORCH_ALLOC_CONF=\"expandable_segments:True\")\n    path = os.path.join(S.LOG_DIR, f\"sglang_{name}.log\")\n    S.log(f\"[{name}] server cmd:\", \" \".join(cmd))\n    fh = open(path, \"w\")\n    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,\n                            start_new_session=True, env=env)\n    return proc, path\n\n\ndef pools(path):\n    txt = open(path, errors=\"ignore\").read()\n    out = {}\n    for key in [\"max_total_num_tokens\", \"max_running_requests\", \"max_mamba_cache_size\", \"context_len\",\n                \"chunked_prefill_size\", \"max_prefill_tokens\"]:\n        m = re.findall(key + r\"\\W{1,3}(\\d+)\", txt)\n        if m:\n            out[key] = int(m[-1])\n    for pat in [r\"KV Cache is allocated\\.[^\\n]*\", r\"Mamba Cache is allocated\\.[^\\n]*\", r\"Memory pool end\\.[^\\n]*\",\n                r\"Load weight end\\.[^\\n]*\", r\"Capture cuda graph end\\.[^\\n]*\", r\"attention_backend[^\\n]{0,80}\"]:\n        m = re.findall(pat, txt)\n        if m:\n            out[pat[:20]] = m[-1][:300]\n    return out\n\n\ndef stop_server(proc):\n    try:\n        os.killpg(proc.pid, 9)\n    except Exception:\n        pass\n    S.sh(\"pkill -9 -f sglang.launch_server; pkill -9 -f 'sglang::'; sleep 8\", timeout=60)\n\n\nasync def bench_cfg(name, radix, steps, mdir, tasks, prefixes):\n    if left() < 12 * 60:\n        S.log(f\"[{name}] skipped: {left() / 60:.1f} min left\")\n        return\n    res = RESULTS[\"configs\"].setdefault(name, {\"radix\": radix})\n    t = time.time()\n    proc, path = launch(mdir, name, radix)\n    ok = S.wait_ready(proc, path, timeout=min(1500, max(60, left() - 120)))\n    res[\"startup_s\"] = round(time.time() - t)\n    res[\"ready\"] = ok\n    res[\"pools\"] = pools(path)\n    S.log(f\"[{name}] ready={ok} in {res['startup_s']}s pools={json.dumps(res['pools'])}\")\n    save()\n    if not ok:\n        res[\"log_tail\"] = S.server_log_tail(path, 80)\n        save()\n        stop_server(proc)\n        return\n    client = S.Client()\n    for label, kw in steps:\n        if left() < 4 * 60:\n            S.log(f\"[{name}] stopping before {label}: time box\")\n            break\n        try:\n            r = await asyncio.wait_for(S.bench_decode(client, prefixes, tasks=tasks, **kw), timeout=max(60, left() - 90))\n            r[\"prompt_tokens_last\"] = client.last_prompt_tokens\n            res[label] = r\n        except Exception as exc:\n            res[label] = {\"error\": f\"{type(exc).__name__}: {exc}\"[:300]}\n            S.log(f\"[{name}] {label} failed: {exc}\")\n        save()\n    res[\"log_metrics\"] = [ln for ln in open(path, errors=\"ignore\").read().splitlines()\n                          if \"Decode batch\" in ln or \"gen throughput\" in ln][-12:]\n    save()\n    stop_server(proc)\n\n\nasync def main_async():\n    tasks = json.load(open(S.comp_file(\"arc-agi_evaluation_challenges.json\")))\n    prefixes = json.load(open(S.find_one(\"/kaggle/input/**/bench_prefixes.json\")))\n    mdir = S.model_dir()\n    await bench_cfg(\"radix_on\", True, [\n        (\"decode_16x40k\", dict(n_seqs=16, gen_tokens=1000)),\n        (\"decode_32x40k_shared\", dict(n_seqs=32, gen_tokens=1000, n_unique=16)),\n    ], mdir, tasks, prefixes)\n    await bench_cfg(\"radix_off\", False, [\n        (\"decode_16x10k\", dict(n_seqs=16, gen_tokens=1000, max_chars=20000)),\n    ], mdir, tasks, prefixes)\n\n\ndef driver_env():\n    \"\"\"Put the host driver's libcuda first (Kaggle images also carry an older compat libcuda).\"\"\"\n    real = \"/usr/local/nvidia/lib64\"\n    if os.path.exists(real + \"/libcuda.so\"):\n        os.environ[\"LD_LIBRARY_PATH\"] = real + \":\" + os.environ.get(\"LD_LIBRARY_PATH\", \"\")\n        os.environ[\"LD_PRELOAD\"] = real + \"/libcuda.so\"\n\n\ndef main():\n    os.makedirs(S.LOG_DIR, exist_ok=True)\n    driver_env()\n    S.log(\"sglang bench start\")\n    try:\n        install()\n        asyncio.run(main_async())\n    except BaseException as exc:\n        import traceback\n        traceback.print_exc()\n        RESULTS[\"fatal\"] = f\"{type(exc).__name__}: {exc}\"[:500]\n    save()\n    S.log(\"sglang bench done:\", json.dumps(RESULTS)[:3000])\n\n\nif __name__ == \"__main__\":\n    main()\n    os._exit(0)\n")

In [ ]:
import subprocess, sys
p = subprocess.Popen([sys.executable, '-u', '/kaggle/working/bench_sglang.py'], stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, cwd='/kaggle/working')
for line in p.stdout:
    print(line, end='', flush=True)
print('exit code', p.wait())